# Récupérer des données (Partie 1) - Bases 2 - 02/03/2026

Dans ce notebook, nous verrons comment récupérer des informations (scraper) depuis Internet, et commencer à constituer votre base de données.

Comme toujours, toutes les cellules peuvent être executées dans l'ordre, mais essayez de prédire la sortie de chaque cellule avant de l'exécuter.
N'hésitez pas à jouer avec le code, le bidouiller pour tester des alternatives — c'est l'un des grands avantages d'utiliser un notebook.

Pour aujourd'hui, nous aurons besoin des packages suivants : 

`conda install requests beautifulsoup4`

`conda install conda-forge::sparqlwrapper`

`conda install conda-forge::polars`

`conda install conda-forge::pyarrow`

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Beautiful Soup

Imaginons que vous vouliez télécharger les données d'un lien `Archive.org` qui pointe vers un PDF.

In [ ]:
# Lien que vous souhaitez télécharger
url = "https://archive.org/details/crash-magazine-02"

# Télécharger le contenu HTML
response = requests.get(url)

# Vérifier si la requête a réussi (code 200)
if response.status_code == 200:
    # Récupérer le contenu HTML
    html_content = response.text
    print(html_content)
else:
    print(f"Erreur {response.status_code} lors du téléchargement du lien.")

Dans ce gros pavé de texte qui décrit la mise en forme de la page, certains des éléments nous intéressent.
Par exemple, la date de publication : 
```
<dl class="metadata-definition">
        <dt>Publication date</dt>
        <dd class="">
          <a href="/search.php?query=date:1984-02">
            <span itemprop="datePublished">1984-02</span>
        </a>
                </dd>
    </dl>
```

Il est ensuite possible d'utiliser BeautifulSoup pour récupérer cet élément. Ce qui nécessite un petit travail manuel d'enquête.

In [ ]:
# Utiliser BeautifulSoup pour analyser le HTML
soup = BeautifulSoup(html_content, 'html.parser')

# Extraire le titre de la page
title = soup.title.text if soup.title else "Titre non trouvé"

# Extraire la date de publication
publication_date_tag = soup.find("dt", string="Publication date")
publication_date = publication_date_tag.find_next("span", {"itemprop": "datePublished"}).text if publication_date_tag else "Date non trouvée"
print(publication_date)

De même, le lien vers le PDF est trouvable dans le corps du texte.

```
<a class="format-summary download-pill" href="/download/crash-magazine-01/Crash_01_Feb_1984.pdf" title="" data-toggle="tooltip" data-placement="auto left" data-container="body" data-original-title="31.8M">
                PDF                <span class="iconochive-download" aria-hidden="true"></span><span class="icon-label sr-only">download</span>              </a>
```

Nous pouvons donc l'extraire d'une façon similaire.

In [ ]:

# lien pour le pdf
soup = BeautifulSoup(html_content, 'html.parser')

# Extraire le lien vers le fichier PDF
pdf_link_tag = soup.find("a", class_="format-summary download-pill", href=lambda href: href and href.endswith(".pdf"))
pdf_link = pdf_link_tag["href"] if pdf_link_tag else "Lien PDF non trouvé"

print(pdf_link)

Pour récupérer des données, nous utilisons encore le module `requests`.

En donnant l'adresse du fichier (que ce soit un fichier audio, image, ou ici PDF), nous pouvons stocker ce qui est récupéré via la structure python `open`:

In [ ]:
response = requests.get("https://archive.org/download/crash-magazine-02/Crash_02_Mar_1984.pdf")

with open('./pdf_sauvegardé.pdf', 'wb') as f:
    f.write(response.content)

### Exercice 1: Une extraction systématique via BeautifulSoup


Imaginons, comme c'est le cas ici, que ce document fasse partie d'une collection dont vous voudriez extraire pour chaque élément, le lien de téléchargement PDF et la date de publication.

Nous avons vu comment le faire pour un document. Faites le désormais pour les 20 premiers numéros de la collection du magazine Crash, tous sur Internet Archive.

- ❓ Observez la structure de l'URL pour éviter de spécifier une par une les adresses.
  - ❗ Faites bien attention au format de l'adresse URL (et au nombre de caractères !)
- ❓ Stockez l'adresse du PDF ainsi que la date de publication pour chaque numéro.
  - ❗ L'adresse du PDF doit avoir un préfixe pour bien être utilisable : https://archive.org Veillez à bien l'ajouter
- ❓ Si le numéro est 1, 5, ou 10, enregistrerz le PDF.
- ❓ Regroupez tout dans une `dataframe` Pandas, et exportez cette dataframe en csv.


In [ ]:
# Votre solution ici

#### (À masquer) Solution de l'excercice 1

In [ ]:
crash_data = {}

for x in range(20):
    this_url = f"https://archive.org/details/crash-magazine-{str(x+1).zfill(2)}"

    # Lien que vous souhaitez télécharger
    url = this_url

    # Télécharger le contenu HTML
    response = requests.get(url)

    # Vérifier si la requête a réussi (code 200)
    if response.status_code == 200:
        html_content = response.text
        soup = BeautifulSoup(html_content, 'html.parser')
        publication_date_tag = soup.find("dt", string="Publication date")
        publication_date = publication_date_tag.find_next("span", {"itemprop": "datePublished"}).text if publication_date_tag else "Date non trouvée"

        pdf_link_tag = soup.find("a", class_="format-summary download-pill", href=lambda href: href and href.endswith(".pdf"))
        pdf_link = pdf_link_tag["href"] if pdf_link_tag else "Lien PDF non trouvé"

        pdf_link = f"https://archive.org{pdf_link}"

        if x in [0, 4, 9]:
            response = requests.get(pdf_link)

            with open(f'./data/magazine_crash_no_{x+1}.pdf', 'wb') as f:
                f.write(response.content)

        crash_data[x] = {'numéro':x+1, 'date de publication':publication_date, 'adresse PDF':pdf_link}

        print(f"Le magazine n°{x} est sorti en: {publication_date} et peut se télécharger à l'adresse : {pdf_link}")
    else:
        print(f"Erreur {response.status_code} lors du téléchargement du lien.")


df = pd.DataFrame.from_dict(crash_data, orient="index")

df.to_csv('./data/magazine_crash_catalogue.csv', index=False)

# 2. SPARQL

Ici, nous allons (re)voir rapidement la structure d'une query SPARQL utile notamment sur Wikipedia/Wikidata.

Nous pouvons utiliser les librairies `SPARPQWrapper` pour pouvoir simplement envoyer et récupérer les résultats d'une query, ainsi que `polars` pour les mettre en forme en DataFrame.

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import polars as pl

Une fois la query écrite, il suffit de la passer au Wrapper, qui va chercher les résultats sur Wikidata et renvoie un JSON, que l'on peut formatter en DataFrame comme ceci.

❓ Avant de compiler la cellule, essayez de deviner ce que fait la query ci-dessous.

In [ ]:
query = """
SELECT ?child ?childLabel
WHERE
{
# ?child  father   Bach
  ?child wdt:P22 wd:Q1339.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }
}
"""

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
results_dataf = pl.json_normalize(results['results']['bindings']).to_pandas()
results_dataf.head()

#### Exercice 2 : Une query Wikidata


En vous basant sur la structure au-dessus, créez une dataframe contenant les tableaux du Musée d'Art et d'Histoire de Genève présents sur Wikidata, et qui ont une image.

- ❓ Aidez vous des propriétés de wikidata pour bien filtrer le lieu, le type de document
- ❓ Ne choisissez que ceux qui ont une image
- ❓ Combien d'entrées y-a-t'il en moins si vous exigez la présence d'une image ?

In [ ]:
# Votre solution ici.

#### (À masquer) Solution de l'exercice 2

Les propriétés RDF Wikidata qui nous intéressent : 

- wdt:P31 "instance of"
- wdt:P18 "image"
- wdt:P195 "collection"

- wd:Q3305213 "painting"
- wd:Q679075 "MAHG"

In [ ]:
query = """
SELECT DISTINCT ?painting ?paintingLabel ?creatorLabel ?image WHERE {
  # Chercher tableau
  ?painting wdt:P31 wd:Q3305213 .

  # collection fait partie du MAHG
  ?painting wdt:P195 wd:Q679075 .

  OPTIONAL { ?painting wdt:P170 ?creator . }
  ?painting wdt:P18 ?image .

               
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
ORDER BY ?paintingLabel
"""

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
results_dataf = pl.json_normalize(results['results']['bindings']).to_pandas()
#results_dataf.head()
results_dataf